# When the data is a graph

MichAl Academy, lesson 3.7.

Run each cell with **Shift+Enter**.

Network flows, Active Directory, process trees and lateral movement are all
graphs. This notebook measures the one idea underneath every graph neural
network, and the two ways it goes wrong.

**On the data.** Nothing suitable ships with scikit-learn and networkx is not in
this image, so the graph is generated here. That is deliberate rather than lazy:
the whole experiment is turning homophily up and down as a dial, which you
cannot do with a fixed dataset.


In [ ]:
import numpy as np
import torch

torch.set_num_threads(1)
np.set_printoptions(precision=4, suppress=True)

N_PER_GROUP, GROUPS, FEAT = 100, 4, 16
NOISE = 4.0


def make_graph(p_in, p_out, seed=0):
    """Nodes in groups. Edges likelier inside a group than between groups."""
    r = np.random.default_rng(seed)
    n = N_PER_GROUP * GROUPS
    label = np.repeat(np.arange(GROUPS), N_PER_GROUP)

    # Each group has a direction in feature space, buried under heavy noise.
    centres = r.normal(size=(GROUPS, FEAT))
    feats = centres[label] + r.normal(scale=NOISE, size=(n, FEAT))

    same = label[:, None] == label[None, :]
    upper = r.random((n, n)) < np.where(same, p_in, p_out)
    adj = np.triu(upper, 1)
    adj = adj + adj.T
    np.fill_diagonal(adj, 0)
    return feats.astype(np.float32), label, adj.astype(np.float32)


def normalise(adj):
    """Standard GCN normalisation: add self loops, then scale by degree."""
    a = adj + np.eye(len(adj), dtype=np.float32)
    dinv = 1.0 / np.sqrt(a.sum(axis=1))
    return (a * dinv[:, None]) * dinv[None, :]


def homophily(label, adj):
    src, dst = np.nonzero(adj)
    return float((label[src] == label[dst]).mean())


def split(n, seed=0):
    idx = np.random.default_rng(seed).permutation(n)
    return idx[: n // 2], idx[n // 2:]


feats, label, adj = make_graph(0.10, 0.01, seed=0)
ahat = normalise(adj)
tr, te = split(len(label))

print(f"nodes {len(label)}, edges {int(adj.sum() // 2)}")
print(f"homophily (fraction of edges joining same-label nodes): {homophily(label, adj):.3f}")


## 1. Averaging over neighbours, before any learning

The whole idea in one line: replace a node's features with the average of its
neighbourhood. Check what that does to how separable the groups are.


In [ ]:
def separation(F, label):
    """Distance between group centres, over the spread within a group."""
    c = np.stack([F[label == g].mean(axis=0) for g in range(GROUPS)])
    within = np.mean([np.linalg.norm(F[label == g] - c[g], axis=1).mean()
                      for g in range(GROUPS)])
    between = np.mean([np.linalg.norm(c[i] - c[j])
                       for i in range(GROUPS) for j in range(GROUPS) if i < j])
    return between / within


print(f"raw features                    : {separation(feats, label):.4f}")
print(f"after one averaging over neighbours: {separation(ahat @ feats, label):.4f}")


The noise on each node is independent; the group's signal is shared. Averaging a
neighbourhood cancels the first and keeps the second. Nothing has been trained.

## 2. The same network, with and without the graph


In [ ]:
class GCN(torch.nn.Module):
    def __init__(self, rounds, hidden=32, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        dims = [FEAT] + [hidden] * rounds
        self.layers = torch.nn.ModuleList(
            [torch.nn.Linear(dims[i], dims[i + 1]) for i in range(rounds)])
        self.out = torch.nn.Linear(dims[-1], GROUPS)

    def forward(self, x, ahat):
        for layer in self.layers:
            x = torch.relu(ahat @ layer(x))     # gather from neighbours, then mix
        return self.out(x)


class MLP(torch.nn.Module):
    """Identical shape, never looks at the graph."""
    def __init__(self, rounds, hidden=32, seed=0):
        super().__init__()
        torch.manual_seed(seed)
        dims = [FEAT] + [hidden] * rounds
        self.layers = torch.nn.ModuleList(
            [torch.nn.Linear(dims[i], dims[i + 1]) for i in range(rounds)])
        self.out = torch.nn.Linear(dims[-1], GROUPS)

    def forward(self, x, ahat):
        for layer in self.layers:
            x = torch.relu(layer(x))
        return self.out(x)


def run(model_cls, feats, label, ahat, tr, te, rounds=2, seed=0, epochs=300, lr=1e-2):
    net = model_cls(rounds, seed=seed)
    opt = torch.optim.Adam(net.parameters(), lr=lr, weight_decay=5e-4)
    lossf = torch.nn.CrossEntropyLoss()
    X, Ah, yt = torch.tensor(feats), torch.tensor(ahat), torch.tensor(label)
    for _ in range(epochs):
        opt.zero_grad()
        lossf(net(X, Ah)[tr], yt[tr]).backward()
        opt.step()
    with torch.no_grad():
        return (net(X, Ah)[te].argmax(1) == yt[te]).float().mean().item()


SEEDS = 3
mlp = float(np.median([run(MLP, feats, label, ahat, tr, te, seed=s) for s in range(SEEDS)]))
gcn = float(np.median([run(GCN, feats, label, ahat, tr, te, seed=s) for s in range(SEEDS)]))
print(f"features only                : {mlp:.4f}")
print(f"same shape, plus neighbours  : {gcn:.4f}")


## 3. Does it need connection to predict the label?

Turn the dial. `p_in` is how likely two nodes in the same group are to be joined,
`p_out` how likely two nodes in different groups are.


In [ ]:
print(f"{'homophily':<12}{'features only':<16}{'with graph':<14}difference")
for p_in, p_out in ((0.10, 0.001), (0.10, 0.01), (0.06, 0.03), (0.05, 0.05), (0.02, 0.08)):
    f2, l2, a2 = make_graph(p_in, p_out, seed=1)
    ah2 = normalise(a2)
    t2, e2 = split(len(l2))
    m = float(np.median([run(MLP, f2, l2, ah2, t2, e2, seed=s) for s in range(SEEDS)]))
    g = float(np.median([run(GCN, f2, l2, ah2, t2, e2, seed=s) for s in range(SEEDS)]))
    print(f"{homophily(l2, a2):<12.3f}{m:<16.4f}{g:<14.4f}{g - m:+.4f}")


Read the last two rows. Where connection predicts almost nothing, the graph
**hurts**: averaging pulls in other groups' signal and you end up worse than
ignoring it. Where neighbours are reliably *different*, it helps again, because
that is information too.

So the question before building any of this is not "do I have a graph" but "does
being connected predict the thing I am trying to label".

## 4. How many rounds?


In [ ]:
for rounds in (1, 2, 3, 4, 6, 8):
    a = float(np.median([
        run(GCN, feats, label, ahat, tr, te, rounds=rounds, seed=s) for s in range(SEEDS)]))
    print(f"rounds {rounds:<3} accuracy {a:.4f}")


It peaks and then collapses. The cause needs no training at all: just average
repeatedly and ask how similar two different nodes have become.


In [ ]:
X, Ah = torch.tensor(feats), torch.tensor(ahat)
for rounds in (1, 2, 4, 8):
    h = X
    with torch.no_grad():
        for _ in range(rounds):
            h = Ah @ h
    hn = h.numpy()
    nrm = hn / (np.linalg.norm(hn, axis=1, keepdims=True) + 1e-12)
    cos = nrm @ nrm.T
    off = cos[~np.eye(len(cos), dtype=bool)]
    print(f"after {rounds} averagings, mean similarity between different nodes: {off.mean():.4f}")


After eight rounds every node is nearly the same vector, so nothing on top can
separate them. Repeated smoothing on a connected graph converges to a constant.
This is called over-smoothing, and it is why practical graph networks are two or
three layers rather than eight.

That is the third time this track has met the same shape: a signal dying through
repeated multiplication by layers in 3.2, by timesteps in 3.6, and by
neighbourhood averaging here.

## 5. What you have

- A graph convolution is neighbour averaging followed by a learned mix, repeated.
- It works because independent noise cancels and shared signal does not, which
  is visible before any training.
- It only pays when connection predicts the label, and it can do actual harm
  when it does not.
- Two or three rounds. Eight turns every node into the same node.

Lesson 3.8 turns things that are not numbers into positions.
